# Day 4: Machine Learning Model for Stress Concentration Factor ($K_t$) Prediction
**Author:** Navneet Namdeo  
**Domain:** Structural Mechanics & Machine Learning  
**Repository:** Machine-Learning-Projects

### Overview
This notebook uses Scikit-Learn to train a Polynomial Regression model on geometric ratios ($r/d$ and $D/d$) of a notched flat plate under tension to predict the theoretical Stress Concentration Factor ($K_t$).

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Generate Synthetic Engineering Dataset (Simulating Peterson's Chart Data)
np.random.seed(42)
n_samples = 150

# Geometric ratios
r_over_d = np.random.uniform(0.05, 0.30, n_samples)  # Notch radius / net width
D_over_d = np.random.uniform(1.10, 2.50, n_samples)  # Plate width ratio

# Physics-based empirical equation with small noise for Kt simulation
# Kt generally decreases as r/d increases
Kt_true = 1.2 + 0.35 / (r_over_d + 0.05) + 0.15 * (D_over_d - 1)
noise = np.random.normal(0, 0.02, n_samples)
Kt_observed = Kt_true + noise

# Build DataFrame
df = pd.DataFrame({
    'r_over_d': r_over_d,
    'D_over_d': D_over_d,
    'Kt': Kt_observed
})

# 2. Features (X) and Target (y)
X = df[['r_over_d', 'D_over_d']]
y = df['Kt']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Transform features to Degree-2 Polynomial Features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# 4. Train Linear Regression Model
model = LinearRegression()
model.fit(X_train_poly, y_train)

# 5. Evaluate Model
y_pred = model.predict(X_test_poly)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=" * 60)
print("     STRESS CONCENTRATION FACTOR (Kt) ML MODEL RESULTS")
print("=" * 60)
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f}")
print(f"R² Accuracy Score               : {r2 * 100:.2f}%")
print("=" * 60)

# 6. Prediction Function for Design Calculations
def predict_stress_concentration(r_d_ratio, D_d_ratio):
    # Pass input as a DataFrame with matching column names to suppress the warning
    input_data = pd.DataFrame([[r_d_ratio, D_d_ratio]], columns=['r_over_d', 'D_over_d'])
    input_poly = poly.transform(input_data)
    predicted_Kt = model.predict(input_poly)[0]
    return predicted_Kt

# Test Prediction
sample_r_d = 0.10
sample_D_d = 1.50
pred_Kt = predict_stress_concentration(sample_r_d, sample_D_d)

print(f"\n[DESIGN TEST] For r/d = {sample_r_d} and D/d = {sample_D_d}:")
print(f"--> Predicted Stress Concentration Factor (Kt): {pred_Kt:.3f}")

     STRESS CONCENTRATION FACTOR (Kt) ML MODEL RESULTS
Root Mean Squared Error (RMSE) : 0.0745
R² Accuracy Score               : 98.92%

[DESIGN TEST] For r/d = 0.1 and D/d = 1.5:
--> Predicted Stress Concentration Factor (Kt): 3.692
